### 문항 1  뒤죽박죽인 날짜 표기를 하나로 통일하기

In [ ]:
# normalize_date(s) -> str | None 함수를 작성할 것

# 위 12가지를 모두 처리할 것 — 변환 불가능하면 None

# 두 자리 연도(24.12.24)는 2024로 해석할 것

# 월·일이 한 자리인 경우(3월 5일)도 03, 05로 채울 것

# 존재하지 않는 날짜(2024-13-45)는 None으로 처리할 것 — 정규표현식만으론 거를 수 없음. 후처리할 것

# 12/24/2024(미국식)와 2024/12/24를 구분할 것

# 각 입력에 대해 어떤 패턴으로 매치됐는지 함께 출력할 것 (기대결과 부분 확인)

# ※ 매칭된 것을 변수로써 꺼내는 방법 (명명 그룹)

# (?P<변수명>정규표현식)

In [19]:
import re
import datetime
import pandas as pd
from collections import Counter

In [16]:
PATTERNS = [
    #2024년 12월 24일 / 2024년 3월 5일
    ('ymd_kr', re.compile(r'(?P<y>\d{4})\s*년\s*(?P<m>\d{1,2})\s*월\s*(?P<d>\d{1,2})\s*일')),

    # 2024.12.24 / 2024-12-24 / 2024/12/24
    ('ymd_sep', re.compile(r'(?P<y>\d{4})[./-](?P<m>\d{1,2})[./-](?P<d>\d{1,2})')),

    # 미국식 12/24/2024
    ('mdy_slash', re.compile(r'(?P<d>\d{1,2})/(?P<m>\d{1,2})/(?P<y>\d{4})')),

    # 두자리 연도 24.12.24
    ('ymd_short', re.compile(r'(?<!\d)(?P<y>\d{1,2})[./-](?P<m>\d{1,2})[./-](?P<d>\d{1,2})(?!\d)'))
]


def normalize_date(s):
    if not s or not str(s).strip():
        return (None, '빈값')

    for tag, pat in PATTERNS:
        m = pat.search(str(s))
        if not m:
            continue

        y, mo, d = int(m['y']), int(m['m']), int(m['d'])
        if tag == 'ymd_short':
            y += 2000 if y < 27 else 1900

        try:
            dt = datetime.date(y, mo, d).strftime('%Y-%m-%d')
        except ValueError:
            return(None, f'{tag} 유효하지 않는 날짜')
        return (dt, tag)
    return (None, '매치 없음')

samples = [
    "2024.12.24",          "2024-12-24",        "2024/12/24",
    "24.12.24",            "2024년 12월 24일",   "2024년 3월 5일",
    "12/24/2024",          "2024.12.24 14:30",  "등록일 : 2024.12.24",
    "2024-13-45",          "작성일 없음",         "",
]

for s in samples:
    value, tag = normalize_date(s)
    print(f'{s:22s} -> {str(value):12s} [{tag}]')


2024.12.24             -> 2024-12-24   [ymd_sep]
2024-12-24             -> 2024-12-24   [ymd_sep]
2024/12/24             -> 2024-12-24   [ymd_sep]
24.12.24               -> 2024-12-24   [ymd_short]
2024년 12월 24일          -> 2024-12-24   [ymd_kr]
2024년 3월 5일            -> 2024-03-05   [ymd_kr]
12/24/2024             -> None         [mdy_slash 유효하지 않는 날짜]
2024.12.24 14:30       -> 2024-12-24   [ymd_sep]
등록일 : 2024.12.24       -> 2024-12-24   [ymd_sep]
2024-13-45             -> None         [ymd_sep 유효하지 않는 날짜]
작성일 없음                 -> None         [매치 없음]
                       -> None         [빈값]


### 문항 2 서버 액세스 로그 파싱과 집계

In [ ]:
# 웹 서버의 액세스 로그를 정규표현식으로 파싱해 분석하시오.

# 명명 그룹 (?P<name>...) 을 사용할 것

# 형식이 깨진 줄은 건너뛰고, 몇 줄을 건너뛰었는지 출력할 것

# 다음 세 가지를 집계해 출력할 것

# 상태코드별 요청 수

# 4xx·5xx가 발생한 경로 상위 5개

# 결과를 access_report.csv로 저장할 것

In [32]:
# 아래 내용을 담은 로그 파일 access.log 를 생성하시오.
# 203.0.113.42 - - [06/Aug/2026:14:22:31 +0900] "GET /list?page=3 HTTP/1.1" 200 5321 "<https://example.com/>" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
# 198.51.100.7 - - [06/Aug/2026:14:22:33 +0900] "POST /api/search HTTP/1.1" 429 118 "-" "python-requests/2.31.0"
# 203.0.113.42 - - [06/Aug/2026:14:22:35 +0900] "GET /detail/9981 HTTP/1.1" 404 209 "<https://example.com/list>" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
# 조건
# 하나의 정규표현식으로 한 줄에서 다음 7개를 추출할 것 
# ip / timestamp / method / path / status / bytes / user_agent

LOG = re.compile(
    r'^(?P<ip>\d{1,3}(?:\.\d{1,3}){3})' # IP -> ?:으로 묶기만 해줌
    r'\s+\S+\s+\S+\s+'
    r'\[(?P<timestamp>[^\]]+)\]\s+'
    r'"(?P<method>[A-Z]+)\s+(?P<path>\S+)[^"]*"\s+'
    r'(?P<status>\d{3})\s+'
    r'(?P<bytes>\d+)\s+'
    r'"[^"]*"\s+'
    r'"(?P<user_agent>[^"]*)"'
)

# 봇으로 의심되는 User-Agent 목록과 그 요청 수 (bot / crawler / spider / python-requests 포함 여부로 판정, 대소문자 무시)
BOT = re.compile('bot|crawler|spider|python-requests', re.IGNORECASE )

def parse_log(path: str) -> tuple[pd.DataFrame, int]:
    rows, skipped = [], 0
    with open(path, encoding='utf-8') as f:
        for line in f.readlines():
            line = line.strip()
            if not line :
                continue
            m = LOG.match(line)
            if not m:
                skipped += 1
                continue
            rows.append(m.groupdict())

    df = pd.DataFrame(rows)
    if not df.empty:
        df['status'] = df['status'].astype(int)
        df['bytes'] = df['bytes'].astype(int)
        df['is_bot'] = df['user_agent'].map(
            lambda x : any(b in x.lower() for b in 'bot|crawler|spider|python-requests'.split('|'))
        )
    return df, skipped

df, skipped = parse_log('access.log')
print(f'파싱 {len(df)}줄 · 건너뜀 {skipped}줄')

print('\n—— 상태코드별 요청 수 ——')
print(df['status'].value_counts().sort_index().to_string())

print('\n—— 4xx·5xx 발생 경로 상위 5 ——')
notcontact = df[df['status'] >= 400]
notcontact_path = notcontact['path'].value_counts().head(5)
print(notcontact_path.to_string() if not notcontact_path.empty else '없음')

print('\n—— 봇 의심 User-Agent ——')
bots = df[df['is_bot']]['user_agent'].value_counts()
print(bots.to_string() if not bots.empty else '없음')
print(f'  봇 요청 비율 {df['is_bot'].mean()*100:.1f}%')

df.to_csv('access_report.csv', index=False, encoding='utf-8-sig')

파싱 3줄 · 건너뜀 0줄

—— 상태코드별 요청 수 ——
status
200    1
404    1
429    1

—— 4xx·5xx 발생 경로 상위 5 ——
path
/api/search     1
/detail/9981    1

—— 봇 의심 User-Agent ——
user_agent
python-requests/2.31.0    1
  봇 요청 비율 33.3%
